# Lab 5 — Bronze: limpeza e tipagem dos dados

## Objetivo

Este laboratório constrói a camada Bronze a partir dos dados da camada Raw.

A Bronze preserva os atributos originais relevantes, mas aplica controles técnicos de qualidade, como correção dos tipos, validação de campos obrigatórios, exclusão de valores impossíveis e tratamento de identificadores duplicados.

Nesta etapa não são criados indicadores de negócio. O objetivo é produzir dados tecnicamente confiáveis para os enriquecimentos da camada Silver.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

pasta_projeto = Path.cwd().resolve()

if not (pasta_projeto / "dados" / "raw").exists():
    for pasta_pai in pasta_projeto.parents:
        if (pasta_pai / "dados" / "raw").exists():
            pasta_projeto = pasta_pai
            break

arquivo_clientes_raw = (
    pasta_projeto
    / "dados"
    / "raw"
    / "sqoop_import"
    / "customers"
    / "customers_from_db.csv"
)

arquivo_transacoes_raw = (
    pasta_projeto
    / "dados"
    / "raw"
    / "sqoop_import"
    / "transactions"
    / "transactions_from_db.csv"
)

pasta_bronze = pasta_projeto / "dados" / "bronze"
pasta_bronze.mkdir(parents=True, exist_ok=True)

arquivo_clientes_bronze = pasta_bronze / "customers.parquet"
arquivo_transacoes_bronze = pasta_bronze / "transactions.parquet"

arquivo_banco = (
    pasta_projeto
    / "dia2_transformacao"
    / "lab05_bronze"
    / "bronze.duckdb"
)

assert arquivo_clientes_raw.exists(), "CSV de clientes não encontrado."
assert arquivo_transacoes_raw.exists(), "CSV de transações não encontrado."

print("Clientes Raw:", arquivo_clientes_raw)
print("Transações Raw:", arquivo_transacoes_raw)
print("Destino Bronze:", pasta_bronze)

Clientes Raw: C:\BigData\bigdata-curso-gabriel\dados\raw\sqoop_import\customers\customers_from_db.csv
Transações Raw: C:\BigData\bigdata-curso-gabriel\dados\raw\sqoop_import\transactions\transactions_from_db.csv
Destino Bronze: C:\BigData\bigdata-curso-gabriel\dados\bronze


In [2]:
conexao = duckdb.connect(str(arquivo_banco))

caminho_clientes = (
    arquivo_clientes_raw.as_posix().replace("'", "''")
)

caminho_transacoes = (
    arquivo_transacoes_raw.as_posix().replace("'", "''")
)

conexao.execute(f"""
    CREATE OR REPLACE VIEW raw_customers AS
    SELECT *
    FROM read_csv_auto(
        '{caminho_clientes}',
        header = true,
        all_varchar = true
    )
""")

conexao.execute(f"""
    CREATE OR REPLACE VIEW raw_transactions AS
    SELECT *
    FROM read_csv_auto(
        '{caminho_transacoes}',
        header = true,
        all_varchar = true
    )
""")

contagem_raw = conexao.execute("""
    SELECT 'Clientes' AS base, COUNT(*) AS linhas
    FROM raw_customers

    UNION ALL

    SELECT 'Transações' AS base, COUNT(*) AS linhas
    FROM raw_transactions
""").df()

contagem_raw

,base,linhas
0,Clientes,9993
1,Transações,100000


In [3]:
diagnostico_clientes = conexao.execute("""
    SELECT
        COUNT(*) AS total_raw,

        SUM(
            CASE WHEN TRY_CAST(customer_id AS BIGINT) IS NULL
                 THEN 1 ELSE 0 END
        ) AS id_invalido,

        SUM(
            CASE WHEN NULLIF(TRIM(name), '') IS NULL
                 THEN 1 ELSE 0 END
        ) AS nome_ausente,

        SUM(
            CASE WHEN TRY_CAST(credit_score AS INTEGER)
                      NOT BETWEEN 300 AND 900
                       OR TRY_CAST(credit_score AS INTEGER) IS NULL
                 THEN 1 ELSE 0 END
        ) AS credit_score_invalido,

        SUM(
            CASE WHEN TRY_CAST(created_at AS DATE) IS NULL
                 THEN 1 ELSE 0 END
        ) AS data_invalida,

        COUNT(*) - COUNT(DISTINCT customer_id)
            AS duplicidades_excedentes
    FROM raw_customers
""").df()

diagnostico_clientes

,total_raw,id_invalido,nome_ausente,credit_score_invalido,data_invalida,duplicidades_excedentes
0,9993,0.0,0.0,0.0,0.0,0


In [4]:
diagnostico_transacoes = conexao.execute("""
    SELECT
        COUNT(*) AS total_raw,

        SUM(
            CASE WHEN TRY_CAST(transaction_id AS BIGINT) IS NULL
                 THEN 1 ELSE 0 END
        ) AS transaction_id_invalido,

        SUM(
            CASE WHEN TRY_CAST(customer_id AS BIGINT) IS NULL
                 THEN 1 ELSE 0 END
        ) AS customer_id_invalido,

        SUM(
            CASE WHEN TRY_CAST(amount AS DOUBLE) <= 0
                       OR TRY_CAST(amount AS DOUBLE) IS NULL
                 THEN 1 ELSE 0 END
        ) AS valor_invalido,

        SUM(
            CASE WHEN TRY_CAST(risk_score AS DOUBLE)
                      NOT BETWEEN 0 AND 100
                       OR TRY_CAST(risk_score AS DOUBLE) IS NULL
                 THEN 1 ELSE 0 END
        ) AS risk_score_invalido,

        SUM(
            CASE WHEN TRY_CAST("timestamp" AS TIMESTAMP) IS NULL
                 THEN 1 ELSE 0 END
        ) AS data_invalida,

        SUM(
            CASE WHEN LOWER(TRIM(is_fraud))
                      NOT IN ('true', 'false', '1', '0')
                       OR is_fraud IS NULL
                 THEN 1 ELSE 0 END
        ) AS fraude_invalida,

        COUNT(*) - COUNT(DISTINCT transaction_id)
            AS duplicidades_excedentes
    FROM raw_transactions
""").df()

diagnostico_transacoes

,total_raw,transaction_id_invalido,customer_id_invalido,valor_invalido,risk_score_invalido,data_invalida,fraude_invalida,duplicidades_excedentes
0,100000,0.0,0.0,0.0,0.0,0.0,0.0,0


In [5]:
conexao.execute("""
    CREATE OR REPLACE TABLE bronze_customers AS

    WITH clientes_tipados AS (
        SELECT
            TRY_CAST(customer_id AS BIGINT) AS customer_id,
            NULLIF(TRIM(name), '') AS name,
            NULLIF(TRIM(cpf), '') AS cpf,
            LOWER(NULLIF(TRIM(email), '')) AS email,
            NULLIF(TRIM(segment), '') AS segment,
            TRY_CAST(credit_score AS INTEGER) AS credit_score,
            TRY_CAST(created_at AS DATE) AS created_at
        FROM raw_customers
    ),

    clientes_validos AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY customer_id
                ORDER BY created_at, name
            ) AS numero_linha
        FROM clientes_tipados
        WHERE customer_id IS NOT NULL
          AND name IS NOT NULL
          AND credit_score BETWEEN 300 AND 900
          AND created_at IS NOT NULL
    )

    SELECT
        customer_id,
        name,
        cpf,
        email,
        segment,
        credit_score,
        created_at
    FROM clientes_validos
    WHERE numero_linha = 1
""")

print("Bronze de clientes criada.")

Bronze de clientes criada.


In [6]:
conexao.execute("""
    CREATE OR REPLACE TABLE bronze_transactions AS

    WITH transacoes_tipadas AS (
        SELECT
            TRY_CAST(transaction_id AS BIGINT) AS transaction_id,
            TRY_CAST(customer_id AS BIGINT) AS customer_id,
            TRY_CAST(amount AS DOUBLE) AS amount,
            LOWER(NULLIF(TRIM(transaction_type), ''))
                AS transaction_type,
            TRY_CAST("timestamp" AS TIMESTAMP)
                AS transaction_timestamp,
            LOWER(NULLIF(TRIM(status), '')) AS status,
            TRY_CAST(risk_score AS DOUBLE) AS risk_score,

            CASE
                WHEN LOWER(TRIM(is_fraud)) IN ('true', '1')
                    THEN TRUE
                WHEN LOWER(TRIM(is_fraud)) IN ('false', '0')
                    THEN FALSE
                ELSE NULL
            END AS is_fraud
        FROM raw_transactions
    ),

    transacoes_validas AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY transaction_id
                ORDER BY transaction_timestamp, customer_id
            ) AS numero_linha
        FROM transacoes_tipadas
        WHERE transaction_id IS NOT NULL
          AND customer_id IS NOT NULL
          AND amount > 0
          AND transaction_type IN (
              'compra',
              'saque',
              'transferencia',
              'pagamento'
          )
          AND transaction_timestamp IS NOT NULL
          AND status IN ('approved', 'declined')
          AND risk_score BETWEEN 0 AND 100
          AND is_fraud IS NOT NULL
    )

    SELECT
        transaction_id,
        customer_id,
        amount,
        transaction_type,
        transaction_timestamp,
        status,
        risk_score,
        is_fraud
    FROM transacoes_validas
    WHERE numero_linha = 1
""")

print("Bronze de transações criada.")

Bronze de transações criada.


In [7]:
comparacao_camadas = conexao.execute("""
    SELECT
        'Clientes' AS base,
        (SELECT COUNT(*) FROM raw_customers) AS linhas_raw,
        (SELECT COUNT(*) FROM bronze_customers) AS linhas_bronze

    UNION ALL

    SELECT
        'Transações' AS base,
        (SELECT COUNT(*) FROM raw_transactions) AS linhas_raw,
        (SELECT COUNT(*) FROM bronze_transactions) AS linhas_bronze
""").df()

comparacao_camadas["linhas_descartadas"] = (
    comparacao_camadas["linhas_raw"]
    - comparacao_camadas["linhas_bronze"]
)

comparacao_camadas

,base,linhas_raw,linhas_bronze,linhas_descartadas
0,Clientes,9993,9993,0
1,Transações,100000,100000,0


In [8]:
validacao_clientes = conexao.execute("""
    SELECT
        COUNT(*) AS total_clientes,
        COUNT(DISTINCT customer_id) AS ids_unicos,
        COUNT(*) - COUNT(DISTINCT customer_id)
            AS ids_duplicados,
        MIN(credit_score) AS menor_credit_score,
        MAX(credit_score) AS maior_credit_score,
        MIN(created_at) AS primeira_data,
        MAX(created_at) AS ultima_data
    FROM bronze_customers
""").df()

validacao_clientes

,total_clientes,ids_unicos,ids_duplicados,menor_credit_score,maior_credit_score,primeira_data,ultima_data
0,9993,9993,0,300,900,2024-08-10,2026-08-10


In [9]:
validacao_transacoes = conexao.execute("""
    SELECT
        COUNT(*) AS total_transacoes,
        COUNT(DISTINCT transaction_id) AS ids_unicos,
        COUNT(*) - COUNT(DISTINCT transaction_id)
            AS ids_duplicados,
        MIN(amount) AS menor_valor,
        MAX(amount) AS maior_valor,
        MIN(risk_score) AS menor_risk_score,
        MAX(risk_score) AS maior_risk_score,
        SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END)
            AS quantidade_fraudes,
        ROUND(
            100.0 * AVG(CASE WHEN is_fraud THEN 1 ELSE 0 END),
            2
        ) AS taxa_fraude_percentual
    FROM bronze_transactions
""").df()

validacao_transacoes

,total_transacoes,ids_unicos,ids_duplicados,menor_valor,maior_valor,menor_risk_score,maior_risk_score,quantidade_fraudes,taxa_fraude_percentual
0,100000,100000,0,10.0,22927.02301,0.003115,99.998694,1833.0,1.83


In [10]:
print("Schema da Bronze de clientes:")
display(
    conexao.execute(
        "DESCRIBE bronze_customers"
    ).df()[["column_name", "column_type"]]
)

print("Schema da Bronze de transações:")
display(
    conexao.execute(
        "DESCRIBE bronze_transactions"
    ).df()[["column_name", "column_type"]]
)

Schema da Bronze de clientes:


,column_name,column_type
0,customer_id,BIGINT
1,name,VARCHAR
2,cpf,VARCHAR
3,email,VARCHAR
4,segment,VARCHAR
5,credit_score,INTEGER
6,created_at,DATE


Schema da Bronze de transações:


,column_name,column_type
0,transaction_id,BIGINT
1,customer_id,BIGINT
2,amount,DOUBLE
3,transaction_type,VARCHAR
4,transaction_timestamp,TIMESTAMP
5,status,VARCHAR
6,risk_score,DOUBLE
7,is_fraud,BOOLEAN


In [11]:
# Remove apenas os arquivos gerados anteriormente pelo próprio Lab 5.
for arquivo in [
    arquivo_clientes_bronze,
    arquivo_transacoes_bronze
]:
    if arquivo.exists():
        arquivo.unlink()

destino_clientes = (
    arquivo_clientes_bronze.as_posix().replace("'", "''")
)

destino_transacoes = (
    arquivo_transacoes_bronze.as_posix().replace("'", "''")
)

conexao.execute(f"""
    COPY bronze_customers
    TO '{destino_clientes}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
""")

conexao.execute(f"""
    COPY bronze_transactions
    TO '{destino_transacoes}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
""")

print("Arquivos da Bronze gravados:")
print("-", arquivo_clientes_bronze)
print("-", arquivo_transacoes_bronze)

Arquivos da Bronze gravados:
- C:\BigData\bigdata-curso-gabriel\dados\bronze\customers.parquet
- C:\BigData\bigdata-curso-gabriel\dados\bronze\transactions.parquet


In [12]:
validacao_arquivos = conexao.execute(f"""
    SELECT
        'Clientes' AS base,
        COUNT(*) AS linhas
    FROM read_parquet('{destino_clientes}')

    UNION ALL

    SELECT
        'Transações' AS base,
        COUNT(*) AS linhas
    FROM read_parquet('{destino_transacoes}')
""").df()

validacao_arquivos

,base,linhas
0,Clientes,9993
1,Transações,100000


In [13]:
conexao.close()

print("Conexão encerrada.")
print("Lab 5 executado com sucesso.")

Conexão encerrada.
Lab 5 executado com sucesso.


## Conclusão

As bases de clientes e transações foram submetidas a controles de qualidade antes de serem armazenadas na camada Bronze. As colunas foram convertidas para tipos adequados, os campos obrigatórios foram verificados e foram aplicadas regras para impedir valores impossíveis.

Nos clientes, foram validados o identificador, o nome, o intervalo do credit score e a data de criação. Nas transações, foram verificados os identificadores, o valor positivo, o tipo de transação, a data, o status, o intervalo do risk score e a conversão explícita do indicador de fraude.

Os identificadores também foram deduplicados, mantendo apenas um registro por cliente ou transação. A comparação entre Raw e Bronze permite identificar de forma objetiva quantos registros foram descartados.

Por fim, as duas tabelas foram persistidas em formato Parquet com compressão ZSTD. O Parquet é mais adequado que o CSV para as próximas etapas por preservar os tipos, utilizar armazenamento colunar e permitir leituras analíticas mais eficientes.